# evalpond quickstart

An eval is a test for an AI feature: a fixed set of tasks with known right answers, so you can tell whether a change helped or hurt.
This notebook runs the whole thing in your browser. No install, no API key needed for the first run.

**Everything here is synthetic.** The documents are generated fake pay stubs and bank statements. Never upload real documents.

Run the cells top to bottom (Runtime > Run all). It takes about a minute.

## 1. Get the tool

In [ ]:
!git clone -q https://github.com/bnovarini/evalpond.git
%cd evalpond
!pip install -q -e .

## 2. Run the guided first run

This makes 60 fake income documents, has three scripted stand-in models take the test, grades them, and builds the report.
The stand-in models are fake on purpose, so this costs nothing.

In [ ]:
!evalpond quickstart --no-open --out site

## 3. Open the report

Start with **Compare runs**. The weak model against the strong one gives a clear verdict. The strong model against its near-identical twin says "no clear difference". That second one is the lesson: small gaps on 60 tasks are noise.

In [ ]:
from IPython.display import HTML, display

display(HTML(open("site/index.html", encoding="utf-8").read()))

In [ ]:
# The report is one file. Download it to open in its own tab, or share it.
try:
    from google.colab import files
    files.download("site/index.html")
except ImportError:
    print("Not on Colab. The report is at site/index.html")

## 4. Optional: test a real model

Skip this if you just wanted to see how it works.

This runs a real model on the pay stub and bank statement tasks (categories A and B, 32 tasks), then adds it to the report next to the stand-in models.
It costs a small amount on your own account. A spending cap is set below, and the run stops if the estimate passes it.

Your key stays in this notebook session. It is never written to a file.
Model names change often, so type one from your provider's current list.

In [ ]:
import getpass
import os

PROVIDER = "anthropic"      # "anthropic" or "openai"
MODEL = ""                  # e.g. paste a current model name from your provider's docs
COST_CAP_USD = 2.0
if MODEL:
    var = "ANTHROPIC_API_KEY" if PROVIDER == "anthropic" else "OPENAI_API_KEY"
    os.environ[var] = getpass.getpass(f"Paste your {PROVIDER} API key (hidden): ")
    cfg = open("models.yaml").read().rstrip() + (
        f"\n  - name: real-model\n    adapter: {PROVIDER}\n    model: {MODEL}\n    api_key_env: {var}\n")
    open("models-real.yaml", "w").write(cfg)
else:
    print("Set MODEL above, then run this cell again.")

In [ ]:
if MODEL:
    !evalpond run --model real-model --models models-real.yaml --judge mock-judge --categories A,B --cost-cap {COST_CAP_USD} --out runs
    !evalpond report runs --out site
    display(HTML(open("site/index.html", encoding="utf-8").read()))

## What next

- Read the [PM guide](https://github.com/bnovarini/evalpond/blob/main/docs/GUIDE.md) for how to read the report.
- Read [why results are noisy](https://github.com/bnovarini/evalpond/blob/main/docs/NOISE.md).
- Write your own tasks: [WRITING_TASKS](https://github.com/bnovarini/evalpond/blob/main/docs/WRITING_TASKS.md).